In [1]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

# ---- Paths ----
TRAIN_PATH = '/kaggle/input/competitions/rogii-wellbore-geology-prediction/train/'
TEST_PATH  = '/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/'
SUB_PATH   = '/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv'

train_hw_files = glob.glob(TRAIN_PATH + '*__horizontal_well.csv')
test_hw_files  = glob.glob(TEST_PATH  + '*__horizontal_well.csv')

def get_well_name(filepath):
    return os.path.basename(filepath).split('__')[0]

# ---- Load Train ----
print('Loading Train Data...')
dfs = []
for f in train_hw_files:
    df = pd.read_csv(f)
    df['well_name'] = get_well_name(f)
    dfs.append(df)
train_df = pd.concat(dfs, ignore_index=True)
train_df = train_df.sort_values(['well_name', 'MD']).reset_index(drop=True)

# ---- Advanced Feature Engineering (Namma Own-ah Add Panra Features) ----
def add_features(df):
    df = df.copy()
    if 'GR' in df.columns:
        # Standard Rolling (Echachal values-ah fill panrom)
        df['GR'] = df.groupby('well_name')['GR'].transform(lambda x: x.fillna(x.median()))
        df['GR_roll3']  = df.groupby('well_name')['GR'].transform(lambda x: x.rolling(3,  min_periods=1).mean())
        df['GR_roll10'] = df.groupby('well_name')['GR'].transform(lambda x: x.rolling(10, min_periods=1).mean())
        df['GR_roll20'] = df.groupby('well_name')['GR'].transform(lambda x: x.rolling(20, min_periods=1).mean())
        df['GR_std10']  = df.groupby('well_name')['GR'].transform(lambda x: x.rolling(10, min_periods=1).std().fillna(0))
        df['GR_diff1']  = df.groupby('well_name')['GR'].transform(lambda x: x.diff(1).fillna(0))
        df['GR_diff5']  = df.groupby('well_name')['GR'].transform(lambda x: x.diff(5).fillna(0))
        
        # PUTHU FEATURES: Geology Lag & Lead
        df['GR_lag1'] = df.groupby('well_name')['GR'].shift(1).fillna(method='bfill')
        df['GR_lag2'] = df.groupby('well_name')['GR'].shift(2).fillna(method='bfill')
        df['GR_lead1'] = df.groupby('well_name')['GR'].shift(-1).fillna(method='ffill')
        df['GR_lead2'] = df.groupby('well_name')['GR'].shift(-2).fillna(method='ffill')
        
        # 3D Spatial Distance Feature
        if all(c in df.columns for c in ['X', 'Y', 'Z']):
            dx = df.groupby('well_name')['X'].diff().fillna(0)
            dy = df.groupby('well_name')['Y'].diff().fillna(0)
            dz = df.groupby('well_name')['Z'].diff().fillna(0)
            df['dist_3d'] = np.sqrt(dx**2 + dy**2 + dz**2)
    return df

train_df = add_features(train_df)
print('✅ Advanced Features Created!')

# ---- All Feature Columns List ----
FEATURE_COLS = ['MD', 'GR', 'GR_roll3', 'GR_roll10', 'GR_roll20', 'GR_std10', 'GR_diff1', 'GR_diff5',
                'GR_lag1', 'GR_lag2', 'GR_lead1', 'GR_lead2', 'dist_3d']
for col in ['X', 'Y', 'Z', 'TVT_input']:
    if col in train_df.columns:
        if col not in FEATURE_COLS:
            FEATURE_COLS.append(col)

# Target Selector
sample_sub = pd.read_csv(SUB_PATH)
SUB_TARGET_COL = sample_sub.columns[1]

TARGET = None
for potential_target in ['tvt', 'TVT', 'target']:
    if potential_target in train_df.columns:
        TARGET = potential_target
        break
if TARGET is None:
    TARGET = train_df.columns[-1]

train_known = train_df.dropna(subset=[TARGET]).reset_index(drop=True)

# ---- K-Fold Cross Validation ----
X = train_known[FEATURE_COLS]
y = train_known[TARGET]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_known))
models = []

print('Training Optimized LightGBM with 5-Fold CV...')
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_trn, y_trn = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=8,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        force_row_wise=True,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_trn, y_trn)
    val_preds = model.predict(X_val)
    oof_preds[val_idx] = val_preds
    models.append(model)
    
    fold_rmse = root_mean_squared_error(y_val, val_preds)
    print(f'Fold {fold+1} RMSE: {fold_rmse:.4f}')

cv_rmse = root_mean_squared_error(y, oof_preds)
print(f'✅ Overall Local CV RMSE: {cv_rmse:.4f}')

# ---- Test Prediction ----
print('Predicting test with advanced features...')
test_dfs = []
for f in test_hw_files:
    df = pd.read_csv(f)
    df['well_name'] = get_well_name(f)
    df['row_index'] = range(len(df))
    test_dfs.append(df)
test_df = pd.concat(test_dfs, ignore_index=True)

test_df = add_features(test_df)
X_test = test_df[FEATURE_COLS]

test_preds = np.zeros(len(test_df))
for model in models:
    test_preds += model.predict(X_test) / len(models)

test_df['id'] = test_df['well_name'] + '_' + test_df['row_index'].astype(str)
test_df[SUB_TARGET_COL] = test_preds

submission = sample_sub[['id']].merge(test_df[['id', SUB_TARGET_COL]], on='id', how='left')
submission[SUB_TARGET_COL] = submission[SUB_TARGET_COL].fillna(test_preds.mean())

submission.to_csv('submission.csv', index=False)
print(f'✅ submission.csv saved! Total rows: {len(submission)}') 

Loading Train Data...
✅ Advanced Features Created!
Training Optimized LightGBM with 5-Fold CV...
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 4073804, number of used features: 17
[LightGBM] [Info] Start training from score 11503.718421
Fold 1 RMSE: 29.3127
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 4073804, number of used features: 17
[LightGBM] [Info] Start training from score 11503.432489
Fold 2 RMSE: 29.7561
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 4073804, number of used features: 17
[LightGBM] [Info] Start training from score 11503.808033
Fold 3 RMSE: 28.9099
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 4073804, number of used features: 17
[LightGBM] [Info] Start training from score 11503.653996
Fold 4 RMSE: 29.9282
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points i